# Exploratory Analysis of Antibody and T-Cell Responses to Influenza Vaccination

**Purpose.** Explore whether T-cell measurements are associated with antibody responses and whether they can help predict which influenza strain produces the strongest antibody response for each subject.

This is an exploratory analysis of the supplied dataset. The goal is to identify interpretable patterns for discussion, not to establish causal or clinical conclusions.

## 1. Load and inspect the data

Each row represents one subject–influenza-strain observation. Antibody measurements are provided before and after vaccination, and the DP, DQ, and DR columns contain T-cell measurements.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Keep the CSV in the same folder as this notebook.
df = pd.read_csv("B-T Cell Challenge.csv")

print("Dataset dimensions:", df.shape)
print("Unique subjects:", df["Subject"].nunique())
print("\nRows per strain:")
display(df["Subtype"].value_counts())

Dataset dimensions: (683, 1661)
Unique subjects: 176

Rows per strain:


Subtype
BYam    175
BVic    174
H1N1    172
H3N2    162
Name: count, dtype: int64

## 2. Prepare the data

Remove the completely empty column, identify the T-cell measurements, and check how many strains are available per subject. T-cell measurements vary across a subject's strain rows, so the strain-specific rows are retained rather than collapsed into one row per person.

In [2]:
df = df.dropna(axis=1, how="all")

tcell_cols = [
    col for col in df.columns
    if col.startswith(("DP.", "DQ.", "DR."))
]

print("Number of T-cell measurement sites:", len(tcell_cols))
print("\nNumber of strains available per subject:")
display(df.groupby("Subject")["Subtype"].nunique().value_counts().sort_index())

print("\nDuplicate subject-strain rows:",
      df.duplicated(subset=["Subject", "Subtype"]).sum())

# Check antibody values before calculating fold change.
print("\nPre-vaccination values <= 0:", (df["PreVac_IgG"] <= 0).sum())
print("Post-vaccination values <= 0:", (df["PostVac_IgG"] <= 0).sum())

Number of T-cell measurement sites: 1656

Number of strains available per subject:


Subtype
2      1
3     19
4    156
Name: count, dtype: int64


Duplicate subject-strain rows: 0

Pre-vaccination values <= 0: 0
Post-vaccination values <= 0: 0


## 3. Quantify antibody response

Use the log2 fold change from pre-vaccination to post-vaccination IgG:

\[
\text{Log2 fold change} = \log_2\left(\frac{\text{PostVac IgG}}{\text{PreVac IgG}}\right)
\]

A value of 1 means a two-fold increase; 0 means no change; and a negative value means antibody levels decreased. Larger values represent larger antibody increases.

In [3]:
df["Log2_Fold_Change"] = np.log2(
    df["PostVac_IgG"] / df["PreVac_IgG"]
)

antibody_summary = (
    df.groupby("Subtype")["Log2_Fold_Change"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)

display(antibody_summary)

,count,mean,median,std
Subtype,,,,
BVic,174,3.65,3.60,1.82
BYam,175,3.53,3.32,1.80
H1N1,172,3.86,3.94,1.91
H3N2,162,3.90,3.96,1.99


## 4. Explore T-cell / antibody associations

For each strain, calculate Spearman correlations between each T-cell site and log2 antibody fold change. Apply Benjamini–Hochberg false-discovery-rate correction across all site-by-strain tests to account for multiple comparisons.

**Direction matters:** the lab notes that lower T-cell values represent stronger T-cell responses. Therefore, a positive correlation between a T-cell measurement and antibody fold change means higher (weaker) T-cell values tend to accompany larger antibody increases.

In [4]:
correlation_results = []

for strain in df["Subtype"].unique():
    strain_data = df[df["Subtype"] == strain]

    for col in tcell_cols:
        paired = strain_data[[col, "Log2_Fold_Change"]].dropna()

        if len(paired) < 20 or paired[col].nunique() < 2:
            continue

        rho, p_value = spearmanr(
            paired[col],
            paired["Log2_Fold_Change"]
        )

        correlation_results.append({
            "Subtype": strain,
            "T_cell_site": col,
            "Spearman_rho": rho,
            "P_value": p_value,
            "N": len(paired)
        })

correlations = pd.DataFrame(correlation_results)

# Correct all site-by-strain tests together.
correlations["Global_FDR"] = multipletests(
    correlations["P_value"],
    method="fdr_bh"
)[1]

significant_correlations = (
    correlations[correlations["Global_FDR"] < 0.05]
    .sort_values("Global_FDR")
)

print("Number of site-by-strain tests:", len(correlations))
print("Associations with global FDR < 0.05:",
      len(significant_correlations))

significant_display = significant_correlations.copy()
significant_display["P_value"] = significant_display["P_value"].map(
    lambda value: f"{value:.2e}"
)
significant_display["Global_FDR"] = significant_display["Global_FDR"].map(
    lambda value: f"{value:.4f}"
)
display(significant_display)

Number of site-by-strain tests: 4935
Associations with global FDR < 0.05: 2


,Subtype,T_cell_site,Spearman_rho,P_value,N,Global_FDR
2056,H3N2,DR.106,0.332582,1.53e-05,162,0.0378
2057,H3N2,DR.107,0.335698,1.26e-05,162,0.0378


### Inspect the H3N2 associations

The two globally significant associations in this analysis are DR.106 and DR.107 for H3N2. Check how closely these two sites track one another and how often the DR.106 measurement is at its maximum observed value.

In [5]:
h3n2 = df[df["Subtype"] == "H3N2"].copy()

print("Spearman correlation between DR.106 and DR.107:")
display(
    h3n2[["DR.106", "DR.107"]].corr(method="spearman").round(4)
)

print("\nDR.106 values for H3N2:")
display(h3n2["DR.106"].value_counts().sort_index())

h3n2_significant = significant_correlations[
    significant_correlations["Subtype"] == "H3N2"
][["T_cell_site", "Spearman_rho", "P_value", "Global_FDR", "N"]].copy()

h3n2_significant["P_value"] = h3n2_significant["P_value"].map(
    lambda value: f"{value:.2e}"
)
h3n2_significant["Global_FDR"] = h3n2_significant["Global_FDR"].map(
    lambda value: f"{value:.4f}"
)
display(h3n2_significant.round({"Spearman_rho": 4}))

Spearman correlation between DR.106 and DR.107:


,DR.106,DR.107
DR.106,1.0000,0.9994
DR.107,0.9994,1.0000



DR.106 values for H3N2:


DR.106
7.7       4
8.2       1
12.0      9
13.0      3
18.0     12
19.0      1
20.0    132
Name: count, dtype: int64

,T_cell_site,Spearman_rho,P_value,Global_FDR,N
2056,DR.106,0.3326,1.53e-05,0.0378,162
2057,DR.107,0.3357,1.26e-05,0.0378,162


**Interpretation.** The positive correlations indicate that higher T-cell measurement values (weaker T-cell responses, according to the dataset description) were associated with larger H3N2 antibody increases. DR.106 and DR.107 are almost perfectly correlated with each other, so they should not be treated as two independent findings. DR.106 is also at 20 for many H3N2 observations, suggesting limited variation / a possible ceiling effect. These are exploratory associations, not evidence of causation.

## 5. Research question: Can T-cell measurements help predict each subject's strongest antibody-response strain?

First calculate each subject's antibody fold change for each strain, then identify the strain with the largest fold change. Some subjects do not have measurements for all four strains; their strongest strain is selected from the strains available for that subject.

In [6]:
antibody_wide = df.pivot(
    index="Subject",
    columns="Subtype",
    values="Log2_Fold_Change"
)

strain_cols = ["H1N1", "H3N2", "BVic", "BYam"]

antibody_wide["Strongest_Strain"] = (
    antibody_wide[strain_cols].idxmax(axis=1)
)

antibody_wide["Strains_Available"] = (
    antibody_wide[strain_cols].notna().sum(axis=1)
)

print("Subjects:", len(antibody_wide))
print("\nStrongest antibody-response strain:")
display(antibody_wide["Strongest_Strain"].value_counts())

print("\nStrains measured per subject:")
display(antibody_wide["Strains_Available"].value_counts().sort_index())

# Check for ties in the maximum antibody fold change.
max_values = antibody_wide[strain_cols].max(axis=1)
number_of_ties = (
    antibody_wide[strain_cols].eq(max_values, axis=0).sum(axis=1) > 1
).sum()

print("Subjects with tied strongest responses:", number_of_ties)

Subjects: 176

Strongest antibody-response strain:


Strongest_Strain
H3N2    58
BVic    46
H1N1    46
BYam    26
Name: count, dtype: int64


Strains measured per subject:


Strains_Available
2      1
3     19
4    156
Name: count, dtype: int64

Subjects with tied strongest responses: 0


In [7]:
# Attach each subject's winning strain to their strain-specific rows.
analysis_df = df.merge(
    antibody_wide["Strongest_Strain"],
    on="Subject"
)

analysis_df["Is_Strongest"] = (
    analysis_df["Subtype"] == analysis_df["Strongest_Strain"]
)

# Baseline: always predict the most common strongest-response strain.
strain_counts = antibody_wide["Strongest_Strain"].value_counts()
baseline_strain = strain_counts.idxmax()
baseline_accuracy = strain_counts.max() / len(antibody_wide)

print("Baseline prediction:", baseline_strain)
print("Baseline accuracy:", round(baseline_accuracy * 100, 2), "%")

Baseline prediction: H3N2
Baseline accuracy: 32.95 %


## 6. Train and evaluate subject-level prediction models

Each subject contributes multiple strain rows. To avoid placing the same subject in both training and test data, use **GroupKFold** with subject ID as the group. For each test subject, score their available strain rows and select the strain with the highest predicted probability of being the strongest antibody-response strain.

The evaluation metric is **top-1 accuracy**: the proportion of subjects for whom the model selects the correct strongest strain. This is an exploratory comparison; the sample is small relative to the number of T-cell features.

In [8]:
model_df = analysis_df[
    ["Subject", "Subtype", "Strongest_Strain", "Is_Strongest"] + tcell_cols
].copy()

def evaluate_top1(estimator, feature_columns):
    X = model_df[feature_columns]
    y = model_df["Is_Strongest"]
    groups = model_df["Subject"]

    predictions = []
    group_cv = GroupKFold(n_splits=5)

    for train_idx, test_idx in group_cv.split(X, y, groups):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y.iloc[train_idx]

        model = estimator()
        model.fit(X_train, y_train)

        probabilities = model.predict_proba(X_test)[:, 1]

        fold_predictions = model_df.iloc[test_idx][
            ["Subject", "Subtype", "Is_Strongest"]
        ].copy()
        fold_predictions["Probability"] = probabilities

        for subject, subject_rows in fold_predictions.groupby("Subject"):
            predicted_strain = subject_rows.loc[
                subject_rows["Probability"].idxmax(), "Subtype"
            ]
            actual_strain = subject_rows.loc[
                subject_rows["Is_Strongest"], "Subtype"
            ].iloc[0]

            predictions.append({
                "Subject": subject,
                "Actual_Strain": actual_strain,
                "Predicted_Strain": predicted_strain,
                "Correct": predicted_strain == actual_strain
            })

    return pd.DataFrame(predictions)

### Compare a small-feature model with models using all T-cell sites

The first model uses DR.106 alone. The second and third use all 1,656 sites. Imputation and scaling (for logistic regression) are fitted within each training fold to avoid using test-fold information during preprocessing.

In [9]:
# Logistic regression with one T-cell site.
single_site_predictions = evaluate_top1(
    lambda: make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=2000)
    ),
    ["DR.106"]
)

# Logistic regression with all T-cell sites.
all_logistic_predictions = evaluate_top1(
    lambda: make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced")
    ),
    tcell_cols
)

# Random Forest with all T-cell sites.
random_forest_predictions = evaluate_top1(
    lambda: make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=200,
            max_depth=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    ),
    tcell_cols
)

model_comparison = pd.DataFrame({
    "Model": [
        "Always predict H3N2 (baseline)",
        "Logistic regression: DR.106",
        "Logistic regression: all T-cell sites",
        "Random Forest: all T-cell sites"
    ],
    "Top-1 accuracy (%)": [
        baseline_accuracy * 100,
        single_site_predictions["Correct"].mean() * 100,
        all_logistic_predictions["Correct"].mean() * 100,
        random_forest_predictions["Correct"].mean() * 100
    ]
})

display(model_comparison.round(2))

,Model,Top-1 accuracy (%)
0,Always predict H3N2 (baseline),32.95
1,Logistic regression: DR.106,28.98
2,Logistic regression: all T-cell sites,32.39
3,Random Forest: all T-cell sites,34.66


### Which strains were easiest for the Random Forest to predict?

The following breakdown uses the out-of-fold predictions from the Random Forest model.

In [10]:
rf_accuracy_by_strain = (
    random_forest_predictions
    .groupby("Actual_Strain")["Correct"]
    .agg(["mean", "count"])
)

rf_accuracy_by_strain["Accuracy (%)"] = rf_accuracy_by_strain["mean"] * 100
rf_accuracy_by_strain = rf_accuracy_by_strain.rename(
    columns={"count": "Subjects"}
)[["Accuracy (%)", "Subjects"]]

display(rf_accuracy_by_strain.round(2))

print("\nStrains predicted by the Random Forest:")
predicted_distribution = (
    random_forest_predictions["Predicted_Strain"]
    .value_counts()
    .rename("Count")
    .to_frame()
)
predicted_distribution["Percentage (%)"] = (
    predicted_distribution["Count"] / len(random_forest_predictions) * 100
)
display(predicted_distribution.round(2))

,Accuracy (%),Subjects
Actual_Strain,,
BVic,19.57,46
BYam,7.69,26
H1N1,23.91,46
H3N2,67.24,58



Strains predicted by the Random Forest:


,Count,Percentage (%)
Predicted_Strain,,
H3N2,87,49.43
H1N1,45,25.57
BVic,38,21.59
BYam,6,3.41


## 7. Summary and limitations

### Main observations

- The dataset contains 176 subjects and 1,656 T-cell measurement sites. Most subjects have measurements for all four strains, but some have fewer.
- H3N2 had the largest number of subjects with the strongest antibody fold change (58 of 176; 32.95%).
- In the site-by-strain correlation analysis, DR.106 and DR.107 for H3N2 were the only associations that remained significant after global false-discovery-rate correction. Their correlations with antibody fold change were positive (approximately 0.33), while lower T-cell values represent stronger T-cell responses.
- DR.106 and DR.107 were almost perfectly correlated with one another, and DR.106 frequently took the value 20 in H3N2 observations. These features limit how independently and confidently the two associations can be interpreted.
- The Random Forest's top-1 accuracy should be compared with the majority-strain baseline. Any improvement is modest and should be treated as preliminary, not as evidence that T-cell measurements can reliably predict an individual's strongest antibody response.

### Limitations

- The sample includes 176 subjects but 1,656 T-cell features, creating a substantial risk of overfitting.
- The four strongest-response classes are imbalanced, and the model may favor the more common H3N2 class.
- Some subjects have fewer than four strain measurements.
- The analyses are exploratory. Correlation does not establish causation, and model performance would need validation on an independent dataset before making strong predictive claims.

### Suggested next step

Discuss the H3N2 DR.106/DR.107 association and the modest prediction results with the lab. A useful next analysis would be to validate selected features and model performance in an independent cohort, if one is available.